# 02 — From perceptron to MLP (with manual backprop)

## Roadmap
1. The single neuron (perceptron) — math and code
2. Activation functions — sigmoid, ReLU, tanh (each with its derivative)
3. Loss functions — MSE vs cross-entropy
4. Manual forward + backward pass for a **2-layer MLP**
5. Gradient descent — actually train the thing
6. Run on a tiny CIFAR subset and watch the loss drop


In [ ]:
# === Bootstrap (safe to re-run) ===
# Installs minimal deps and downloads tiny CIFAR-10 subset for any notebook
# that needs it. The from-scratch notebooks deliberately avoid importing the
# project package so they run anywhere (Colab, plain Python, Jupyter).
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "numpy", "matplotlib", "pillow", "torchvision"], check=True)
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)
print("numpy", np.__version__)


## 1. The single neuron

A neuron is a dot product followed by a non-linear activation:

$$y = \sigma(w \cdot x + b)$$

Without the non-linearity, stacking neurons does **nothing** — composition of linear functions is still linear. The non-linearity is what makes deep networks expressive.


In [ ]:
def neuron(x, w, b, activation):
    return activation(x @ w + b)

# A "or-gate-ish" neuron with sigmoid activation
def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))

w = np.array([5.0, 5.0]); b = -2.5
for x in [[0,0], [0,1], [1,0], [1,1]]:
    print(x, "→", round(float(neuron(np.array(x), w, b, sigmoid)), 3))


## 2. Activation functions (and why we need their derivatives)

| Name | Formula | Derivative | Use |
|---|---|---|---|
| **Sigmoid** | $\sigma(z) = \frac{1}{1+e^{-z}}$ | $\sigma(z)(1-\sigma(z))$ | Binary output |
| **Tanh** | $\tanh(z)$ | $1 - \tanh^2(z)$ | Hidden, classical |
| **ReLU** | $\max(0, z)$ | 0 if z<0 else 1 | Hidden, modern default |
| **Softmax** | $e^{z_i} / \sum_j e^{z_j}$ | (computed via cross-entropy) | Multi-class final |


In [ ]:
def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))
def sigmoid_grad(z): s = sigmoid(z); return s * (1 - s)
def relu(z): return np.maximum(0, z)
def relu_grad(z): return (z > 0).astype(z.dtype)
def tanh(z): return np.tanh(z)
def tanh_grad(z): return 1 - np.tanh(z) ** 2

# Plot the three
zs = np.linspace(-5, 5, 200)
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, (fn, gn, name) in zip(axes, [(sigmoid, sigmoid_grad, "sigmoid"),
                                       (tanh, tanh_grad, "tanh"),
                                       (relu, relu_grad, "ReLU")]):
    ax.plot(zs, fn(zs), label=name)
    ax.plot(zs, gn(zs), label="derivative", linestyle="--")
    ax.set_title(name); ax.legend(); ax.grid(True)
plt.show()


## 3. Loss functions

A loss function tells you "how wrong" the model is on a training example. The training loop nudges weights to make this smaller.

**MSE (regression):** $\mathcal{L} = \frac{1}{N} \sum_i (\hat{y}_i - y_i)^2$

**Cross-entropy (classification):** $\mathcal{L} = -\sum_i y_i \log \hat{y}_i$

Cross-entropy combined with softmax has a beautifully simple gradient: $\hat{y} - y$. This is why every classifier uses this combination.


In [ ]:
def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)   # numerical stability
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

def cross_entropy(y_true_onehot, y_pred):
    # add small eps for numerical safety
    return -float((y_true_onehot * np.log(y_pred + 1e-9)).sum(axis=-1).mean())

# example
probs = softmax(np.array([[2.0, 0.1, 0.2]]))
print("softmax:", probs.round(3), "sum:", probs.sum())
print("CE if true class=0:", cross_entropy(np.array([[1,0,0]]), probs))


## 4. The 2-layer MLP — implement forward AND backward by hand

Two linear layers with a ReLU between them, ending in softmax + cross-entropy. We will derive every gradient and verify against NumPy.

Architecture: `input (D)  →  Linear(D, H)  →  ReLU  →  Linear(H, C)  →  Softmax  →  CE loss`


In [ ]:
class MLP:
    """2-layer MLP, pure NumPy, with manual backprop."""

    def __init__(self, in_dim, hidden, out_dim, lr=0.1, seed=0):
        rng = np.random.default_rng(seed)
        # He init for ReLU layer; small Gaussian for the second.
        self.W1 = rng.standard_normal((in_dim, hidden)).astype(np.float32) * np.sqrt(2.0/in_dim)
        self.b1 = np.zeros(hidden, dtype=np.float32)
        self.W2 = rng.standard_normal((hidden, out_dim)).astype(np.float32) * 0.01
        self.b2 = np.zeros(out_dim, dtype=np.float32)
        self.lr = lr

    def forward(self, X):
        # X: (N, D)
        self.X = X
        self.z1 = X @ self.W1 + self.b1            # (N, H)
        self.a1 = np.maximum(0, self.z1)           # ReLU
        self.z2 = self.a1 @ self.W2 + self.b2      # (N, C)
        self.p  = softmax(self.z2)                 # (N, C)
        return self.p

    def loss(self, Y_onehot):
        return -float((Y_onehot * np.log(self.p + 1e-9)).sum(axis=-1).mean())

    def backward(self, Y_onehot):
        # Softmax + CE gradient is simply (p - y) / N
        N = self.X.shape[0]
        dz2 = (self.p - Y_onehot) / N                  # (N, C)
        dW2 = self.a1.T @ dz2                          # (H, C)
        db2 = dz2.sum(axis=0)
        da1 = dz2 @ self.W2.T                          # (N, H)
        dz1 = da1 * (self.z1 > 0)                      # ReLU' = step
        dW1 = self.X.T @ dz1                           # (D, H)
        db1 = dz1.sum(axis=0)
        # SGD step
        self.W1 -= self.lr * dW1; self.b1 -= self.lr * db1
        self.W2 -= self.lr * dW2; self.b2 -= self.lr * db2

print("MLP class compiled.")


## 5. Train on CIFAR-10 (tiny, flattened to a vector)

A flat MLP on raw image pixels is **not** the right architecture for images — that's why NB 03+04 exist. But it converges enough to demonstrate the training loop.


In [ ]:
# Load a tiny CIFAR-10 subset.
# torchvision is used ONLY to fetch the dataset bytes — no torch tensors,
# no torch.nn. We immediately convert to NumPy.
from torchvision import datasets
from pathlib import Path
DATA_ROOT = Path("./_cifar_cache")
DATA_ROOT.mkdir(exist_ok=True)

train_ds = datasets.CIFAR10(root=DATA_ROOT, train=True,  download=True)
test_ds  = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True)

def to_numpy(ds, n):
    X = np.stack([np.asarray(ds[i][0], dtype=np.float32) / 255.0 for i in range(n)])
    y = np.array([ds[i][1] for i in range(n)], dtype=np.int64)
    return X, y

X_train, y_train = to_numpy(train_ds, 2000)
X_test,  y_test  = to_numpy(test_ds,  500)
CLASSES = ("plane","car","bird","cat","deer","dog","frog","horse","ship","truck")
print("X_train", X_train.shape, "y_train", y_train.shape)
print("X_test ", X_test.shape,  "y_test ", y_test.shape)


In [ ]:
# Flatten 32x32x3 images to vectors of 3072, take 2 classes (cat, dog) for speed
mask_tr = (y_train == 3) | (y_train == 5)
mask_te = (y_test  == 3) | (y_test  == 5)
X_tr = X_train[mask_tr].reshape(-1, 32*32*3)
y_tr = (y_train[mask_tr] == 5).astype(np.int64)   # 1 if dog, 0 if cat
X_te = X_test[mask_te].reshape(-1, 32*32*3)
y_te = (y_test[mask_te] == 5).astype(np.int64)

# Center the data — helps the MLP a lot
mean = X_tr.mean(axis=0); std = X_tr.std(axis=0) + 1e-8
X_tr = (X_tr - mean) / std
X_te = (X_te - mean) / std

Y_tr = np.eye(2)[y_tr].astype(np.float32)

mlp = MLP(in_dim=32*32*3, hidden=128, out_dim=2, lr=0.05)
hist = []
for epoch in range(20):
    mlp.forward(X_tr)
    L = mlp.loss(Y_tr)
    mlp.backward(Y_tr)
    p_te = mlp.forward(X_te)
    acc_te = (p_te.argmax(axis=-1) == y_te).mean()
    hist.append((L, acc_te))
    print(f"epoch {epoch+1:2d}  train_loss={L:.4f}  test_acc={acc_te:.3f}")

plt.figure(figsize=(8,3))
plt.plot([h[0] for h in hist], label="train loss")
plt.twinx().plot([h[1] for h in hist], color="orange", label="test acc")
plt.title("MLP on flattened CIFAR (cat vs dog)"); plt.show()


## 6. Why this isn't enough

A flattened image loses all spatial structure. Two pixels that are neighbours in the image are arbitrarily far apart in the flat vector — the MLP has to *learn* that they're related. Convolution gives the model that knowledge for free.

**Next:** implement `conv2d` from scratch.
